# 💻 01: Environment & Hardware Verification

## Goal
Ensure all high-performance data libraries are installed and confirm that the **Apple Silicon (M3 Pro) GPU** is accessible for vector computations.

---

## Technology Stack

| Category | Library | Version | Role |
|---|---|---|---|
| **Storage** | PyArrow | 23.0.1 | Parquet I/O — up to 10x compression vs CSV |
| **Data Frameworks** | Pandas | 3.0.1 | DataFrame operations and feature engineering |
| **Data Frameworks** | Polars | 1.39.3 | High-speed data processing via Rust backend |
| **Numerical** | NumPy | 2.4.3 | Backbone for 384-dimensional book embeddings |
| **Numerical** | SciPy | 1.17.1 | Scientific computing and sparse matrix support |
| **ML / NLP** | Scikit-learn | 1.8.0 | Classical ML algorithms and preprocessing |
| **ML / NLP** | Sentence-Transformers | 5.3.0 | Semantic embedding generation |
| **Vector Search** | FAISS | 1.13.2 | High-speed nearest neighbor search |
| **GPU Backend** | PyTorch + MPS | 2.11.0 | Apple Silicon GPU acceleration via Metal |

---

## Hardware Verification Results

| Test | Result | Detail |
|---|---|---|
| **MPS Availability** | ✅ SUPPORTED | Apple M3 Pro GPU accessible via PyTorch MPS backend |
| **Matrix Multiply** | ✅ 0.0940s | 4,000 × 4,000 matrix multiplication on M3 GPU |
| **Compute Device** | ✅ `mps:0` | Operations confirmed running on local GPU |

> **Note:** Matrix operations for cosine similarity and FAISS indexing will leverage the MPS backend throughout the pipeline, significantly reducing inference time compared to CPU-only execution.

In [1]:
# ============================================================
# CELL 1 — Full environment verification
# ============================================================
import sys
import os

print(f"Python version  : {sys.version}")
print("-" * 30)

# IO & Storage (Parquet Support)
import pyarrow as pa
import pandas as pd
import polars as pl
print(f"Storage Engine  : PyArrow {pa.__version__} (Parquet Enabled)")
print(f"Data Frameworks : Pandas {pd.__version__}, Polars {pl.__version__}")

# Numerical & Vector Computing
import numpy as np
import scipy
print(f"Numerical Stack : Numpy {np.__version__}, Scipy {scipy.__version__}")

# Machine Learning & Embeddings
import sklearn
import sentence_transformers
import faiss
print(f"ML - NLP Stack  : Scikit-learn {sklearn.__version__}, Sentence-Transformers {sentence_transformers.__version__}")
print(f"Vector Search   : FAISS {faiss.__version__}")

# GPU Acceleration (Apple Silicon)
try:
    import torch
    mps_status = "SUPPORTED" if torch.backends.mps.is_available() else "NOT SUPPORTED"
    print(f"Compute Backend : PyTorch {torch.__version__} | MPS Hardware: {mps_status}")
except ImportError:
    print("Compute Backend : PyTorch not found")

print("\n✅ All systems green for Hybrid Recommender development.")

Python version  : 3.11.14 (main, Oct  9 2025, 16:16:55) [Clang 17.0.0 (clang-1700.4.4.1)]
------------------------------
Storage Engine  : PyArrow 23.0.1 (Parquet Enabled)
Data Frameworks : Pandas 3.0.1, Polars 1.39.3
Numerical Stack : Numpy 2.4.3, Scipy 1.17.1
ML - NLP Stack  : Scikit-learn 1.8.0, Sentence-Transformers 5.3.0
Vector Search   : FAISS 1.13.2
Compute Backend : PyTorch 2.11.0 | MPS Hardware: SUPPORTED

✅ All systems green for Hybrid Recommender development.


In [2]:
# ============================================================
# CELL 2 — MPS Compute Performance Test
# ============================================================
import torch
import time

if torch.backends.mps.is_available():
    device = torch.device("mps")
    
    # Create large matrices for a meaningful test
    size = 4000 
    x = torch.randn(size, size, device=device)
    y = torch.randn(size, size, device=device)
    
    start = time.time()
    z = torch.matmul(x, y)
    torch.mps.synchronize() # Wait for GPU to finish
    end = time.time()
    
    print(f"✅ MPS Device Verified: {z.device}")
    print(f"Matrix Multiply ({size}x{size}) took: {end-start:.4f} seconds on M3 GPU")
else:
    print("⚠️ MPS not available. Falling back to CPU.")

✅ MPS Device Verified: mps:0
Matrix Multiply (4000x4000) took: 0.2467 seconds on M3 GPU
